# FormulaLM — PROMPT 9: Evaluate Fine-Tuned QLoRA v1 Adapter & Error Analysis

Loads the trained **QLoRA v1** adapter (`checkpoint-200`, best `eval_loss = 0.004240`) onto `Qwen/Qwen3-1.7B` in non-thinking mode (`temperature=0`, `max_tokens=96`, `seed=42`) and evaluates it on BOTH held-out test sets:
1. `results/ft_v1_real` — `data/test_real/test_real_DRAFT.csv` (80 hand-written realistic requests)
2. `results/ft_v1_synthetic` — `data/processed/test_synthetic.jsonl` (500 unseen-schema, unseen-phrasing examples)

Then compares **Base 0-shot vs Base 5-shot vs Fine-tuned v1** (overall, by category, by difficulty) and clusters all v1 failures into 6 actionable clusters with 5 examples each.

**Hardware:** Colab T4 GPU (~2–3 minutes total).

In [1]:
# 1. Verify CUDA T4 GPU and mount Google Drive (where qlora_v1/adapter was saved)
import pathlib, torch

assert torch.cuda.is_available(), 'No CUDA GPU detected! Switch kernel to T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped (will fall back to Hugging Face adapter if not on Drive):', exc)

GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# 2. Clone / pull FormulaLM, install eval deps, and ensure test_synthetic.jsonl exists
import getpass, os, pathlib, subprocess, sys

REPO_DIR = pathlib.Path('/content/FormulaLM')
if not REPO_DIR.exists():
    token = os.environ.get('GITHUB_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            token = None
    if not token:
        token = getpass.getpass('GitHub PAT for abubakarp789/FormulaLM (hidden, empty if public): ').strip()
    url = f'https://{token}@github.com/abubakarp789/FormulaLM.git' if token else 'https://github.com/abubakarp789/FormulaLM.git'
    subprocess.run(['git', 'clone', url, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

%cd /content/FormulaLM
if '/content/FormulaLM/src' not in sys.path:
    sys.path.insert(0, '/content/FormulaLM/src')

%pip install -q formulas openpyxl pandas pyyaml requests tqdm accelerate peft "transformers>=4.51.0" "huggingface_hub>=0.30.0"

if not pathlib.Path('data/processed/test_synthetic.jsonl').is_file():
    !python -m src.data.build_dataset --workers 2

/content/FormulaLM


In [3]:
# 3. Load Qwen/Qwen3-1.7B + QLoRA v1 adapter and evaluate on BOTH test sets
import pathlib
from src.eval.run_eval import TransformersBackend, evaluate

MODEL_ID = 'Qwen/Qwen3-1.7B'
REVISION = '70d244cc86ccca08cf5af4e1e306ecf908b1ad5e'

# Auto-locate adapter: Drive -> local outputs -> Hugging Face Hub
CANDIDATE_ADAPTERS = [
    '/content/drive/MyDrive/FormulaLM/qlora_v1/adapter',
    'outputs/qlora_v1/adapter',
]
ADAPTER_PATH = next((p for p in CANDIDATE_ADAPTERS if pathlib.Path(p).is_dir()), 'abubakar56/formulalm-qwen3-1.7b-lora')
print('Loading adapter from:', ADAPTER_PATH, flush=True)

backend = TransformersBackend(
    MODEL_ID,
    adapter_path=ADAPTER_PATH,
    revision=REVISION,
    load_in_4bit=False,
    enable_thinking=False,
)

for run_name, dataset in [
    ('ft_v1_real', 'data/test_real/test_real_DRAFT.csv'),
    ('ft_v1_synthetic', 'data/processed/test_synthetic.jsonl'),
]:
    print(f'=== Running {run_name} ===', flush=True)
    m = evaluate(
        backend=backend,
        dataset_path=dataset,
        run_name=run_name,
        seed=42,
        max_tokens=96,
        few_shot=0,
        batch_size=8,
        resume=True,
    )
    ov = m['overall']
    print(
        f"{run_name}: n={ov['n']} | syntax={100*ov['syntax_valid_rate']:.1f}% | "
        f"exec_acc={100*ov['execution_accuracy']:.1f}% | "
        f"exact={100*ov['exact_match_rate']:.1f}% | "
        f"raw_fmt={100*ov['raw_format_ok_rate']:.1f}% | "
        f"avg_lat={ov['avg_latency_ms']:.1f}ms",
        flush=True,
    )

Loading adapter from: /content/drive/MyDrive/FormulaLM/qlora_v1/adapter


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
W1123 22:59:08.356000 58061 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1123 22:59:08.419000 58061 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

=== Running ft_v1_real ===


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


ft_v1_real: n=80 | syntax=100.0% | exec_acc=91.2% | exact=91.2% | raw_fmt=100.0% | avg_lat=789.9ms
=== Running ft_v1_synthetic ===
ft_v1_synthetic: n=500 | syntax=99.6% | exec_acc=97.6% | exact=97.0% | raw_fmt=100.0% | avg_lat=794.7ms


In [4]:
# 4. Produce full Base vs Fine-Tuned v1 comparison table + 6-cluster error analysis
!python -m src.eval.compare_and_cluster \
  --ft-synthetic results/ft_v1_synthetic \
  --ft-real results/ft_v1_real \
  --ft-label "Fine-tuned v1" \
  --examples-per-cluster 5 \
  --out results/ft_v1_comparison.md \
  --json-out results/ft_v1_comparison.json

# Comparison & Error Analysis: Base vs Fine-tuned v1

## 1. Overall Comparison

| Test Set | System | N | Syntax Valid | **Exec. Accuracy** | Δ vs 0-shot | Exact Match | Raw Format | Avg Latency |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| `test_synthetic` | Base zero-shot | 500 | 71.8% | **44.6%** | — | 36.6% | 100.0% | 814.0 ms |
| `test_synthetic` | Base 5-shot | 500 | 85.0% | **60.2%** | +15.6 pp | 51.2% | 100.0% | 1728.2 ms |
| `test_synthetic` | **Fine-tuned v1** | 500 | 99.6% | **97.6%** | +53.0 pp | 97.0% | 100.0% | 794.7 ms |
| `test_real` | Base zero-shot | 80 | 81.2% | **50.0%** | — | 42.5% | 100.0% | 651.7 ms |
| `test_real` | Base 5-shot | 80 | 85.0% | **51.2%** | +1.2 pp | 41.2% | 100.0% | 1576.2 ms |
| `test_real` | **Fine-tuned v1** | 80 | 100.0% | **91.2%** | +41.2 pp | 91.2% | 100.0% | 789.9 ms |

## 2A. Execution Accuracy by Category (`test_synthetic`)

| Category | N | Base 0-shot | Base 5-shot | **Fine-tuned v1** | Δ vs 0-shot |
|---|---:|---:|---:|---:|---:|
|

In [5]:
# 5. Push lightweight evaluation results (results/ft_v1_*) straight back to GitHub
PUSH_TO_GITHUB = True
if PUSH_TO_GITHUB:
    !git config user.name "abubakarp789"
    !git config user.email "abubakarp789@gmail.com"
    !git add results/ft_v1_*
    !git commit -m "results: add QLoRA v1 evaluation and 6-cluster error analysis"
    !git pull --rebase origin main
    !git push origin main

[main c7e0971] results: add QLoRA v1 evaluation and 6-cluster error analysis
 10 files changed, 2313 insertions(+)
 create mode 100644 results/ft_v1_comparison.json
 create mode 100644 results/ft_v1_comparison.md
 create mode 100644 results/ft_v1_real/metrics.json
 create mode 100644 results/ft_v1_real/predictions.jsonl
 create mode 100644 results/ft_v1_real/report.md
 create mode 100644 results/ft_v1_real/run_config.json
 create mode 100644 results/ft_v1_synthetic/metrics.json
 create mode 100644 results/ft_v1_synthetic/predictions.jsonl
 create mode 100644 results/ft_v1_synthetic/report.md
 create mode 100644 results/ft_v1_synthetic/run_config.json
error: cannot pull with rebase: You have unstaged changes.
error: Please commit or stash them.
Enumerating objects: 17, done.
Counting objects: 100% (17/17), done.
Delta compression using up to 2 threads
Compressing objects: 100% (15/15), done.
Writing objects: 100% (15/15), 52.19 KiB | 4.35 MiB/s, done.
Total 15 (delta 5), reused 0 (delta